# Matching dan Merge Data ABIDE I

Notebook ini mencocokkan data fenotipik ABIDE I dengan citra sMRI `.nii.gz` berdasarkan `SUB_ID`. Proses yang dilakukan adalah normalisasi ID, validasi keterbacaan header NIfTI, dan **inner join**. Subjek tanpa pasangan pada salah satu sumber tidak masuk ke master dataset.

Notebook ini tidak melakukan preprocessing citra, cleaning klinis, imputasi, feature engineering, atau pemodelan.

## 1. Setup dan konfigurasi path

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nibabel as nib
from IPython.display import display

PHENOTYPIC_DIR = Path(r"C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\fenotipik")
SMRI_DIR = Path(r"C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\all_anat_T1w")
PHENOTYPIC_PATH = PHENOTYPIC_DIR / "Phenotypic_V1_0b.csv"
MERGE_DIR = Path(r"C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\merge_dataset")
MASTER_PATH = MERGE_DIR / "master_dataset.csv"
FILE_PATTERN = "*.nii.gz"

for required_path in [PHENOTYPIC_PATH, SMRI_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(f"Input tidak ditemukan: {required_path}")
MERGE_DIR.mkdir(parents=True, exist_ok=True)
print(f"CSV fenotipik : {PHENOTYPIC_PATH}")
print(f"Folder sMRI   : {SMRI_DIR}")
print(f"Output master : {MASTER_PATH}")

CSV fenotipik : C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\fenotipik\Phenotypic_V1_0b.csv
Folder sMRI   : C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\all_anat_T1w
Output master : C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\merge_dataset\master_dataset.csv


## 2. Fungsi normalisasi dan validasi

Nama citra memiliki pola seperti `sub-0051456_T1w.nii.gz`, sedangkan CSV menggunakan `SUB_ID=51456`. Normalisasi dilakukan melalui angka ID, bukan posisi baris. Jika nama file tidak memuat pola `sub-...`, file tersebut dicatat sebagai tidak match.

In [2]:
SUBJECT_PATTERN = re.compile(r"sub-([0-9]+)", re.IGNORECASE)

def normalize_sub_id(value):
    if pd.isna(value):
        return np.nan
    digits = re.findall(r"\d+", str(value))
    if not digits:
        return np.nan
    return str(int(digits[-1]))

def extract_image_id(path):
    match = SUBJECT_PATTERN.search(path.name)
    return match.group(1).lstrip("0") or "0" if match else np.nan

def validate_nifti_header(path):
    try:
        image = nib.load(str(path), mmap=True)
        return True, "", tuple(image.shape), tuple(image.header.get_zooms()[:3])
    except Exception as error:
        return False, f"{type(error).__name__}: {error}", np.nan, np.nan

## 3. Membaca fenotipik dan memeriksa ID

In [3]:
phenotypic = pd.read_csv(PHENOTYPIC_PATH, low_memory=False)
if "SUB_ID" not in phenotypic.columns:
    raise KeyError("Kolom SUB_ID tidak tersedia pada CSV fenotipik.")
phenotypic["merge_sub_id"] = phenotypic["SUB_ID"].map(normalize_sub_id)
phenotypic_id_summary = pd.DataFrame({
    "metric": ["rows", "unique_SUB_ID", "missing_merge_sub_id", "duplicate_merge_sub_id"],
    "value": [len(phenotypic), phenotypic["SUB_ID"].nunique(dropna=True),
              phenotypic["merge_sub_id"].isna().sum(),
              phenotypic["merge_sub_id"].duplicated(keep=False).sum()],
})
display(phenotypic_id_summary)
if phenotypic["merge_sub_id"].duplicated().any():
    raise ValueError("Terdapat SUB_ID duplikat setelah normalisasi; matching dihentikan agar tidak menggandakan subjek.")

,metric,value
0,rows,1112
1,unique_SUB_ID,1112
2,missing_merge_sub_id,0
3,duplicate_merge_sub_id,0


## 4. Inventarisasi dan validasi file sMRI

Validasi hanya membuka header/objek NIfTI dengan memory mapping. File corrupt atau tidak bisa dibaca tidak dihapus; file tersebut hanya diberi `valid_nifti=False` dan tidak digunakan pada inner join.

In [4]:
image_paths = sorted(SMRI_DIR.glob(FILE_PATTERN), key=lambda path: path.name.lower())
image_rows = []
for path in image_paths:
    valid, error, shape, spacing = validate_nifti_header(path)
    image_rows.append({
        "image_file": path.name,
        "image_path": str(path.resolve()),
        "merge_sub_id": extract_image_id(path),
        "valid_nifti": valid,
        "nifti_error": error,
        "image_shape": str(shape) if valid else np.nan,
        "voxel_spacing": str(spacing) if valid else np.nan,
    })
images = pd.DataFrame(image_rows)
print(f"Total file sMRI: {len(images)}")
print(f"File NIfTI valid: {int(images['valid_nifti'].sum()) if not images.empty else 0}")
print(f"File NIfTI invalid: {int((~images['valid_nifti']).sum()) if not images.empty else 0}")
print(f"ID citra kosong: {int(images['merge_sub_id'].isna().sum()) if not images.empty else 0}")
if not images.empty:
    display(images.head())
    invalid_images = images.loc[~images["valid_nifti"]]
    if not invalid_images.empty:
        print("Daftar file yang gagal divalidasi:")
        display(invalid_images)
if images["merge_sub_id"].duplicated().any():
    duplicate_images = images[images["merge_sub_id"].duplicated(keep=False)]
    display(duplicate_images)
    raise ValueError("Terdapat lebih dari satu file citra untuk merge_sub_id yang sama; matching dihentikan.")

Total file sMRI: 1102
File NIfTI valid: 1102
File NIfTI invalid: 0
ID citra kosong: 0


,image_file,image_path,merge_sub_id,valid_nifti,nifti_error,image_shape,voxel_spacing
0,sub-0050002_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,50002,True,,"(176, 256, 256)","(np.float32(1.0500001), np.float32(1.0507812),..."
1,sub-0050003_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,50003,True,,"(176, 256, 256)","(np.float32(1.0500001), np.float32(1.0507812),..."
2,sub-0050004_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,50004,True,,"(176, 256, 256)","(np.float32(1.0500001), np.float32(1.0507812),..."
3,sub-0050005_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,50005,True,,"(176, 256, 256)","(np.float32(1.0500001), np.float32(1.0507812),..."
4,sub-0050006_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,50006,True,,"(176, 256, 256)","(np.float32(1.0500001), np.float32(1.0507812),..."


## 5. Inner join dan pembentukan master dataset

Hanya citra dengan header NIfTI valid dan ID yang berhasil diekstrak yang digunakan. Kolom bantu `merge_sub_id` dipertahankan agar proses matching dapat diaudit; kolom ini bukan fitur model.

In [5]:
valid_images = images.loc[images["valid_nifti"] & images["merge_sub_id"].notna()].copy()
phenotypic_for_merge = phenotypic.copy()
image_for_merge = valid_images.copy()
master_dataset = phenotypic_for_merge.merge(
    image_for_merge,
    on="merge_sub_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_phenotypic", "_smri"),
)
master_dataset.to_csv(MASTER_PATH, index=False)
print(f"Master dataset tersimpan: {MASTER_PATH}")
display(master_dataset.head())

Master dataset tersimpan: C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR\dataset\merge_dataset\master_dataset.csv


,SITE_ID,SUB_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,SEX,HANDEDNESS_CATEGORY,HANDEDNESS_SCORES,FIQ,...,EYE_STATUS_AT_SCAN,AGE_AT_MPRAGE,BMI,merge_sub_id,image_file,image_path,valid_nifti,nifti_error,image_shape,voxel_spacing
0,CALTECH,51456,Caltech_0051456,1,4,55.4,1,R,NaN,126.0,...,2,NaN,NaN,51456,sub-0051456_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,True,,"(176, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(..."
1,CALTECH,51457,Caltech_0051457,1,4,22.9,1,Ambi,NaN,107.0,...,2,NaN,NaN,51457,sub-0051457_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,True,,"(176, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(..."
2,CALTECH,51458,Caltech_0051458,1,1,39.2,1,R,NaN,93.0,...,2,NaN,NaN,51458,sub-0051458_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,True,,"(176, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(..."
3,CALTECH,51459,Caltech_0051459,1,1,22.8,1,R,NaN,106.0,...,2,NaN,NaN,51459,sub-0051459_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,True,,"(176, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(..."
4,CALTECH,51460,Caltech_0051460,1,1,34.6,2,Ambi,NaN,133.0,...,2,NaN,NaN,51460,sub-0051460_T1w.nii.gz,C:\Users\Lenovo\OneDrive\Documents\TUGAS AKHIR...,True,,"(176, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(..."


## 6. Rekap matching dan distribusi kelas

In [6]:
phenotypic_ids = set(phenotypic["merge_sub_id"].dropna())
image_ids = set(valid_images["merge_sub_id"].dropna())
master_ids = set(master_dataset["merge_sub_id"].dropna())
matching_summary = pd.DataFrame({
    "metric": ["rows_phenotypic", "unique_phenotypic_ids", "total_smri_files", "valid_smri_files", "unique_valid_smri_ids", "matched_ids", "phenotypic_without_valid_smri", "valid_smri_without_phenotypic", "master_rows"],
    "value": [len(phenotypic), len(phenotypic_ids), len(images), int(images["valid_nifti"].sum()), len(image_ids), len(master_ids), len(phenotypic_ids - image_ids), len(image_ids - phenotypic_ids), len(master_dataset)],
})
display(matching_summary)
if "DX_GROUP" in master_dataset.columns:
    class_distribution = master_dataset["DX_GROUP"].value_counts(dropna=False).rename_axis("DX_GROUP").reset_index(name="count")
    class_distribution["percentage"] = class_distribution["count"] / len(master_dataset) * 100 if len(master_dataset) else 0
    display(class_distribution)
    print("Konvensi umum ABIDE I: DX_GROUP=1 ASD dan DX_GROUP=2 kontrol/TD; verifikasi tetap mengacu pada dokumentasi dataset.")
else:
    print("Kolom DX_GROUP tidak tersedia; distribusi kelas tidak dapat dihitung.")
print("\nKolom master dataset:")
print(list(master_dataset.columns))

,metric,value
0,rows_phenotypic,1112
1,unique_phenotypic_ids,1112
2,total_smri_files,1102
3,valid_smri_files,1102
4,unique_valid_smri_ids,1102
5,matched_ids,1102
6,phenotypic_without_valid_smri,10
7,valid_smri_without_phenotypic,0
8,master_rows,1102


,DX_GROUP,count,percentage
0,2,571,51.814882
1,1,531,48.185118


Konvensi umum ABIDE I: DX_GROUP=1 ASD dan DX_GROUP=2 kontrol/TD; verifikasi tetap mengacu pada dokumentasi dataset.

Kolom master dataset:
['SITE_ID', 'SUB_ID', 'FILE_ID', 'DX_GROUP', 'DSM_IV_TR', 'AGE_AT_SCAN', 'SEX', 'HANDEDNESS_CATEGORY', 'HANDEDNESS_SCORES', 'FIQ', 'VIQ', 'PIQ', 'FIQ_TEST_TYPE', 'VIQ_TEST_TYPE', 'PIQ_TEST_TYPE', 'ADI_R_SOCIAL_TOTAL_A', 'ADI_R_VERBAL_TOTAL_BV', 'ADI_RRB_TOTAL_C', 'ADI_R_ONSET_TOTAL_D', 'ADI_R_RSRCH_RELIABLE', 'ADOS_MODULE', 'ADOS_TOTAL', 'ADOS_COMM', 'ADOS_SOCIAL', 'ADOS_STEREO_BEHAV', 'ADOS_RSRCH_RELIABLE', 'ADOS_GOTHAM_SOCAFFECT', 'ADOS_GOTHAM_RRB', 'ADOS_GOTHAM_TOTAL', 'ADOS_GOTHAM_SEVERITY', 'SRS_VERSION', 'SRS_RAW_TOTAL', 'SRS_AWARENESS', 'SRS_COGNITION', 'SRS_COMMUNICATION', 'SRS_MOTIVATION', 'SRS_MANNERISMS', 'SCQ_TOTAL', 'AQ_TOTAL', 'COMORBIDITY', 'CURRENT_MED_STATUS', 'MEDICATION_NAME', 'OFF_STIMULANTS_AT_SCAN', 'VINELAND_RECEPTIVE_V_SCALED', 'VINELAND_EXPRESSIVE_V_SCALED', 'VINELAND_WRITTEN_V_SCALED', 'VINELAND_COMMUNICATION_STANDARD', 'VI